##Advanced read operation by reading csv file format (practice)

In [0]:
table_schema="custid int,name string,salary decimal(10,2),join_date string,corrupt_record string"
adv_read=spark.read.schema(table_schema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/malformeddata/malformeddata.txt",header=False,inferSchema=False,comment="#",quote="'",mode="permissive",columnNameOfCorruptRecord='corrupt_record',dateFormat="yyyy-MM-dd")
display(adv_read)

In [0]:
from pyspark.sql.functions import to_date, col

table_schema="custid int,name string,salary decimal(10,2),join_date_str string,corrupt_record string"
adv_read1=spark.read.schema(table_schema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/malformeddata/malformeddata1.txt",header=False,inferSchema=False,sep=',',comment="#",mode="permissive",quote="'",escape='~',columnNameOfCorruptRecord='corrupt_record',multiLine=True,ignoreLeadingWhiteSpace=True,ignoreTrailingWhiteSpace=True,nullValue='na',nanValue=-1,modifiedAfter='2025-12-12',maxCharsPerColumn='100',lineSep='~')
display(adv_read1)

#using py code, created new column name as "join_date" and mention the date format present in the csv data and drop the older column, so spark will display the date with correct format as per spark standards (yyyy-mm-dd)
'''
df_fixed = adv_read1.withColumn(
    "join_date",
    to_date(col("join_date_str"), "yyyy-dd-MM")
).drop("join_date_str")

display(df_fixed)
'''

In [0]:
#read json into spark dataframe
#jsonschema="id int,name string,amt decimal(10,2),dop date,corruptedrecord string" #id string will give id value in the second record
jsondf1=spark.read.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/samplejson",pathGlobFilter="*json.txt",recursiveFileLookup=True,columnNameOfCorruptRecord=True,dateFormat="yyyy-dd-MM",prefersDecimal=True,primitivesAsString=True)
jsondf1.printSchema()
display(jsondf1)

jsondf2=spark.read.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/samplejson/simple_json2.txt",allowComments=True,lineSep="~",allowSingleQuotes=True,allowUnquotedFieldNames=True,samplingRatio=1)
display(jsondf2)

jsonschema="id int,name string,amt decimal(10,2),dop date,corruptrecord string"
jsondf3=spark.read.schema(jsonschema).json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/samplejson/simple_json_multiline3.txt",columnNameOfCorruptRecord="corruptrecord",allowComments=True,multiLine=True,dateFormat="yyyy-dd-MM")
display(jsondf3)

In [0]:
#Implement the strategy of Schema Evolution using option mergeSchema
#step-1: read the source data and save it in target location in orc/parquet format
day1df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/sampledate_intervals/source1.txt",header=True,inferSchema=True)
day1df.write.mode("append").format("orc").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput")
day2df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/sampledate_intervals/source2.txt",header=True,inferSchema=True)
day2df.write.mode("append").format("orc").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput")
day3df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/sampledate_intervals/source3.txt",header=True,inferSchema=True)
day3df.write.mode("append").format("orc").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput")

#step-2: as the data is evolved with updated schema, use mergeSchema option to read the data
finaldaydf=spark.read.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput",mergeSchema=True)
display(finaldaydf)

#partition the data based on amt column
finaldaydf.write.partitionBy("amt").mode("overwrite").format("orc").save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput")
display(spark.read.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target/orcoutput/amt=20.3/"))